# Optimización de Política Relativa de Grupo (GRPO) con LoRA/QLoRA usando TRL — en un Notebook Gratuito de Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huggingface/trl/blob/main/examples/notebooks/grpo_trl_lora_qlora.ipynb)


![trl banner](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/trl_banner_dark.png)

Ajusta fácilmente **Modelos de Lenguaje a Gran Escala (LLMs)** o **Modelos de Visión y Lenguaje (VLMs)** con **LoRA** o **QLoRA** usando la librería [**Transformers Reinforcement Learning (TRL)**](https://github.com/huggingface/trl) de Hugging Face y la Optimización de Política Relativa de Grupo (GRPO) — todo dentro de un **notebook gratuito de Google Colab** con una **GPU T4**.

Gracias a las **optimizaciones de memoria y entrenamiento integradas en TRL**, incluyendo LoRA, cuantización, gradient checkpointing y kernels de atención optimizados, es posible **ajustar un modelo de 7B en una T4 gratuita** con una **reducción del consumo de memoria de ~7×** en comparación con el entrenamiento ingenuo en FP16.

- [Repositorio TRL en GitHub](https://github.com/huggingface/trl) — ¡danos una estrella para apoyar el proyecto!
- [Ejemplos Oficiales de TRL](https://huggingface.co/docs/trl/example_overview)
- [Tutoriales de la Comunidad](https://huggingface.co/docs/trl/community_tutorials)


## Conceptos Clave

- **GRPO**: Un algoritmo de aprendizaje por refuerzo que optimiza una política comparando múltiples respuestas generadas para el mismo prompt y actualizando el modelo según sus recompensas relativas, sin necesitar un modelo de valor separado.
- **LoRA**: Actualiza únicamente unos pocos parámetros de bajo rango, reduciendo el coste de entrenamiento y el uso de memoria.
- **QLoRA**: Una versión cuantizada de LoRA que permite ajustar modelos aún más grandes en GPUs pequeñas.
- **TRL**: La librería de Hugging Face que hace que el ajuste fino y el aprendizaje por refuerzo sean sencillos y eficientes.

Aprende a realizar **GRPO (Optimización de Política Relativa de Grupo)** con **LoRA/QLoRA** usando **TRL**.


Esta tabla muestra cómo la **activación progresiva de técnicas de eficiencia** afecta al **uso de memoria** y al **rendimiento de entrenamiento** en distintas configuraciones de hardware.
Las técnicas van desde el entrenamiento ingenuo en FP16 hasta **LoRA, cuantización, kernels Liger, paged_adamw_8bit y gradient checkpointing**.

| Configuración | LoRA | Cuant. | Liger | Optimizador | Grad. Ckpt | attn_impl  | VRAM (T4) GB | VRAM (A100-40GB)| VRAM (A100-80GB) | Tokens/s (T4) | Tokens/s (A100-40GB) | Tokens/s (A100-80GB) | Estado (T4) |
|--------------|------|-------|-------|-----------|------------|-----------|---------------|----------------|---------|---------|---------------|------------------|-------------|
| **Peor (FP16 ingenuo)** | ❌ | ❌ | ❌ | AdamW | ❌  | eager | OOM | OOM | 62 GB | - | - | 0.06 it/s | ❌ |
| **Mejor (todas las optimizaciones)** | ✅ | ✅ | ✅ | paged_adamw_8bit | ✅ | sdpa  | 9.2 GB | 9.6 GB | 9.6 GB | 0.01 it/s | 0.03 it/s | 0.04 it/s | ✅ |

Con todas las técnicas de eficiencia activadas, **el uso de memoria en Colab T4 se reduce ~7×**, lo que hace posible **ajustar un modelo de 7B en Colab gratuito** donde el entrenamiento ingenuo en FP16 fallaría.

> Se observa una pequeña pérdida en la velocidad de entrenamiento, pero **la reducción de VRAM es el factor clave habilitador**. Para un entrenamiento más rápido en hardware compatible, también se puede aprovechar **vLLM**.

> 💡 Nota: Para una comparación justa, el número de generaciones y el tamaño del batch no se modificaron.


## Instalación de dependencias

Instalaremos **TRL** con el extra **PEFT**, que garantiza que todas las dependencias principales como **Transformers** y **PEFT** (un paquete para el ajuste fino eficiente en parámetros, p. ej., LoRA/QLoRA) estén incluidas. Además, instalaremos **trackio** para registrar y monitorizar nuestros experimentos, **bitsandbytes** para habilitar la cuantización de LLMs, reduciendo el consumo de memoria tanto en inferencia como en entrenamiento, y **liger-kernel** para un entrenamiento más eficiente.


In [ ]:
!pip install -Uq "trl[peft]" bitsandbytes trackio math_verify liger-kernel

### Iniciar sesión en Hugging Face

Inicia sesión en tu cuenta de **Hugging Face** para guardar tu modelo ajustado, hacer seguimiento de los resultados de tus experimentos directamente en el Hub o acceder a modelos restringidos. Puedes encontrar tu **token de acceso** en la [página de configuración de tu cuenta](https://huggingface.co/settings/tokens).


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Cargar el Dataset

En este paso, cargamos el dataset [**AI-MO/NuminaMath-TIR**](https://huggingface.co/datasets/AI-MO/NuminaMath-TIR) desde el Hub de Hugging Face usando la librería `datasets`.
Este dataset se centra en el **razonamiento matemático**, con problemas que requieren soluciones lógicas paso a paso.
Al ajustar un modelo que todavía no muestra capacidades sólidas de razonamiento, puede aprender a **generar pasos de razonamiento estructurado**, mejorando tanto la **precisión** como la **interpretabilidad** del modelo en tareas matemáticas.

Por eficiencia, cargaremos únicamente una **pequeña parte de la partición de entrenamiento**:


In [ ]:
from datasets import load_dataset

dataset_name = 'AI-MO/NuminaMath-TIR'
train_dataset = load_dataset(dataset_name, split='train[:5%]')

Revisemos la estructura del dataset


In [ ]:
print(train_dataset)

Dataset({
    features: ['problem', 'solution', 'messages'],
    num_rows: 3622
})


Veamos una muestra:


In [ ]:
print(train_dataset[0])

{'problem': 'What is the coefficient of $x^2y^6$ in the expansion of $\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  Express your answer as a common fraction.', 'solution': "To determine the coefficient of \\(x^2y^6\\) in the expansion of \\(\\left(\\frac{3}{5}x - \\frac{y}{2}\\right)^8\\), we can use the binomial theorem.\n\nThe binomial theorem states:\n\\[\n(a + b)^n = \\sum_{k=0}^{n} \\binom{n}{k} a^{n-k} b^k\n\\]\n\nIn this case, \\(a = \\frac{3}{5}x\\), \\(b = -\\frac{y}{2}\\), and \\(n = 8\\).\n\nWe are interested in the term that contains \\(x^2y^6\\). In the general term of the binomial expansion:\n\\[\n\\binom{8}{k} \\left(\\frac{3}{5}x\\right)^{8-k} \\left(-\\frac{y}{2}\\right)^k\n\\]\n\nTo get \\(x^2\\), we need \\(8 - k = 2\\), thus \\(k = 6\\).\n\nSubstituting \\(k = 6\\) into the expression:\n\\[\n\\binom{8}{6} \\left(\\frac{3}{5}x\\right)^{8-6} \\left(-\\frac{y}{2}\\right)^6 = \\binom{8}{6} \\left(\\frac{3}{5}x\\right)^2 \\left(-\\frac{y}{2}\\right)^6\n\\]\n\nNow, we wi

Adaptaremos nuestro dataset a un formato conversacional usando un prompt de sistema personalizado, guiando al LLM para que genere tanto el razonamiento paso a paso como la respuesta final.


In [ ]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant  "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process is enclosed strictly within <think> and </think> tags. "
    "After closing </think>, the assistant MUST provide the final answer in plain text."
)


def make_conversation(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["problem"]},
        ],
    }

train_dataset = train_dataset.map(make_conversation)

Echemos un vistazo a un ejemplo:


In [ ]:
print(train_dataset[0]['prompt'])

[{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant  first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process is enclosed strictly within <think> and </think> tags. After closing </think>, the assistant MUST provide the final answer in plain text.', 'role': 'system'}, {'content': 'What is the coefficient of $x^2y^6$ in the expansion of $\\left(\\frac{3}{5}x-\\frac{y}{2}\\right)^8$?  Express your answer as a common fraction.', 'role': 'user'}]


Eliminaremos las columnas `messages` y `problem`, ya que solo necesitamos la columna personalizada `prompt` y `solution` para verificar la respuesta generada.


In [ ]:
train_dataset = train_dataset.remove_columns(['messages', 'problem'])
print(train_dataset)

Dataset({
    features: ['solution', 'prompt'],
    num_rows: 3622
})


## Cargar el modelo y configurar LoRA/QLoRA

A continuación, elige tu **modelo preferido**. Todas las opciones han sido probadas en **instancias gratuitas de Colab**.

> 💡 Nota: Algunos modelos, como Qwen2.5 y Qwen3, se sabe que han sido preentrenados con datos que mejoran su rendimiento matemático. Ten cuidado al seleccionar el modelo adecuado para el entrenamiento y asegúrate de obtener resultados de ajuste significativos ([fuente](https://thinkingmachines.ai/blog/lora/)).


In [ ]:
# Select one model below by uncommenting the line you want to use 👇
## Qwen
model_id, output_dir = "Qwen/Qwen2-7B-Instruct", "t4-Qwen2-7B-Instruct-GRPO"                             # ✅ ~9.2GB VRAM
# model_id, output_dir = "unsloth/qwen3-14b-unsloth-bnb-4bit", "qwen3-14b-unsloth-bnb-4bit-GRPO"         # ⚠️ OOM with this config; fits if GRPO params are reduced
# model_id, output_dir = "Qwen/Qwen3-8B", "Qwen3-8B-GRPO"                                                # ✅ ~9.9GB VRAM
# model_id, output_dir = "Qwen/Qwen2.5-7B-Instruct", "Qwen2.5-7B-Instruct-GRPO"                          # ✅ ~9.2GB VRAM

## Llama
# model_id, output_dir = "meta-llama/Llama-3.2-3B-Instruct", "Llama-3.2-3B-Instruct-GRPO"             # ✅ ~5.7GB VRAM
# model_id, output_dir = "meta-llama/Llama-3.1-8B-Instruct", "Llama-3.1-8B-Instruct-GRPO"             # ✅ ~9.5GB VRAM

## LFM2.5
# model_id, output_dir = "LiquidAI/LFM2.5-1.2B-Instruct", "LFM2.5-1.2B-Instruct-GRPO"                                   # ✅ ~1.12 GB VRAM

Este notebook puede usarse con dos métodos de ajuste fino. Por defecto, está configurado para **QLoRA**, que incluye cuantización mediante `BitsAndBytesConfig`. Si prefieres usar **LoRA** estándar sin cuantización, simplemente comenta la configuración de `BitsAndBytesConfig` (el entrenamiento sin cuantización consume más memoria).

Carguemos el modelo seleccionado usando `transformers`, configurando QLoRA mediante `bitsandbytes` (puedes eliminarlo si usas LoRA). No necesitamos configurar el tokenizador, ya que el trainer se encarga de eso automáticamente.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",                   # Change to Flash Attention if GPU has support
    dtype="float32",                          # Change to bfloat16 if GPU has support
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,                        # Load the model in 4-bit precision to save memory
        bnb_4bit_compute_dtype=torch.float16,     # Data type used for internal computations in quantization
        bnb_4bit_use_double_quant=True,           # Use double quantization to improve accuracy
        bnb_4bit_quant_type="nf4"                 # Type of quantization. "nf4" is recommended for recent LLMs
    )
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

La siguiente celda define LoRA (o QLoRA si es necesario). Al entrenar con LoRA/QLoRA, usamos un **modelo base** (el seleccionado anteriormente) y, en lugar de modificar sus pesos originales, ajustamos un **adaptador LoRA**, una capa ligera que permite un entrenamiento eficiente y amigable con la memoria. Los **`target_modules`** especifican qué partes del modelo (p. ej., capas de atención o proyección) serán adaptadas por LoRA durante el ajuste fino.


In [ ]:
from peft import LoraConfig

# You may need to update `target_modules` depending on the architecture of your chosen model.
# For example, different LLMs might have different attention/projection layer names.
peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
)

## Entrenar el modelo

GRPO requiere **funciones de recompensa** para guiar el proceso de aprendizaje. Por comodidad, podemos cargar directamente recompensas predefinidas desde `trl.rewards`, que ya incluye una [colección de recompensas listas para usar](https://huggingface.co/docs/trl/rewards).

Si quieres crear tus propias funciones de recompensa personalizadas para enseñar al modelo, una función de recompensa es simplemente una función de Python que toma las completaciones generadas y devuelve una lista de flotantes. Por ejemplo, la siguiente función, que usamos en este notebook, recompensa las completaciones que siguen correctamente el formato `<think>`:

In [ ]:
def think_format_reward(completions: list[list[dict[str, str]]], **kwargs) -> list[float]:
    pattern = r"^<think>(?!.*<think>)(.*?)</think>.*$"
    completion_contents = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, content, re.DOTALL | re.MULTILINE) for content in completion_contents]
    return [1.0 if match else 0.0 for match in matches]

En este notebook, usaremos tanto `think_format_reward`, que recompensa las completaciones que siguen correctamente el formato `<think>`, como `reasoning_accuracy_reward`, que evalúa la corrección de la solución del modelo al problema matemático. Juntas, estas recompensas guían al modelo para generar **razonamiento estructurado** mientras produce **respuestas precisas**.

In [ ]:
from trl.rewards import think_format_reward, reasoning_accuracy_reward

Configuraremos **GRPO** usando `GRPOConfig`, manteniendo los parámetros al mínimo para que el entrenamiento pueda ejecutarse en una instancia gratuita de Colab. Puedes ajustar estos parámetros si tienes acceso a más recursos. Para una lista completa de parámetros disponibles y sus descripciones, consulta la [documentación de TRL GRPOConfig](https://huggingface.co/docs/trl/grpo_trainer#trl.GRPOConfig).

> 💡 Nota: TRL admite el uso de **vLLM** para la generación durante el entrenamiento con GRPO, lo que puede acelerar significativamente el entrenamiento. Sin embargo, aumenta el uso de VRAM ya que hay un proceso vLLM separado activo para gestionar la generación. En este notebook, no habilitamos vLLM porque estamos usando **QLoRA**, que actualiza los pesos del modelo vLLM cuantizado en cada paso. Habilitar vLLM en esta configuración puede causar problemas de precisión en los pesos y dificultar la convergencia. La configuración incluye los parámetros de vLLM por si quieres experimentar. Aprende más sobre la integración de vLLM en TRL [aquí](https://huggingface.co/docs/trl/main/en/vllm_integration).


In [ ]:
from trl import GRPOConfig

# Configure training arguments using GRPOConfig
training_args = GRPOConfig(
    # Training schedule / optimization
    learning_rate=2e-5,                                     # Learning rate for the optimizer
    #num_train_epochs=1,
    max_steps=500,                                          # Number of dataset passes. For full trainings, use `num_train_epochs` instead

    # Parameters that control GRPO training (you can adapt them)
    per_device_train_batch_size = 8,
    max_completion_length=256, # default: 256               # Max completion length produced during training
    num_generations=8, # default: 8                         # Number of generations produced during trainig for comparison

    # Optimizations
    optim = "paged_adamw_8bit",                             # Optimizer
    use_liger_kernel=True,                                  # Enable Liger kernel optimizations for faster training

    # Parameters related to reporting and saving
    output_dir=output_dir,                                  # Where to save model checkpoints and logs
    logging_steps=10,                                       # Log training metrics every N steps
    report_to="trackio",                                    # Experiment tracking tool
    trackio_space_id=output_dir,                            # HF Space where the experiment tracking will be saved
    log_completions=False,                                  # Return model completions during training

    # Hub integration
    push_to_hub=True,                                       # Automatically push the trained model to the Hugging Face Hub
                                                            # The model will be saved under your Hub account in the repository named `output_dir`
    # vLLM params
    #use_vllm=False,                                        # Activate vLLM training for faster training
    #vllm_mode='colocate',
    #vllm_gpu_memory_utilization=0.1,
    #vllm_enable_sleep_mode=True
)

Configura el `GRPOTrainer` pasando los `training_args` definidos anteriormente. Para mantener el uso de memoria bajo, no estamos usando un dataset de evaluación, pero puedes incluir uno si lo deseas. También proporcionamos las funciones de recompensa importadas anteriormente para guiar el proceso de entrenamiento.


In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[think_format_reward, reasoning_accuracy_reward],
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
)

Mostrar estadísticas de memoria antes del entrenamiento


In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
6.773 GB of memory reserved.


¡Y a entrenar!


El entrenamiento en una T4 en Colab con la configuración definida en este notebook tarda unas 13 horas. Si solo estás experimentando, puedes probar la siguiente tarea más rápida ([fuente](https://huggingface.co/learn/llm-course/en/chapter12/5)):

```python
dataset = load_dataset("mlabonne/smoltldr")

# Función de recompensa
ideal_length = 50

def reward_len(completions, **kwargs):
    return [-abs(ideal_length - len(completion)) for completion in completions]
```


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


* Trackio project initialized: huggingface
* Trackio metrics will be synced to Hugging Face Dataset: sergiopaniego/t4-Qwen2-7B-Instruct-GRPO-dataset
* Creating new space: https://huggingface.co/spaces/sergiopaniego/t4-Qwen2-7B-Instruct-GRPO
* View dashboard by going to: https://sergiopaniego-t4-Qwen2-7B-Instruct-GRPO.hf.space/


* Created new run: sergiopaniego-1766143600


Step,Training Loss
10,0.027900
20,-0.011600
30,0.021500
40,0.033400
50,0.039400
60,0.010300
70,0.048200
80,0.067300
90,0.030600
100,0.064000


* Run finished. Uploading logs to Trackio (please wait...)


Mostrar estadísticas de memoria después del entrenamiento


In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

47228.679 seconds used for training.
787.14 minutes used for training.
Peak reserved memory = 8.832 GB.
Peak reserved memory for training = 2.059 GB.
Peak reserved memory % of max memory = 59.915 %.
Peak reserved memory for training % of max memory = 13.968 %.


El procedimiento de entrenamiento genera tanto los logs de entrenamiento estándar como los logs de **trackio**, que nos ayudan a monitorizar el progreso del entrenamiento. Los resultados de ejemplo tendrían el siguiente aspecto:


<img src="https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/grpo-qlora-notebook-trackio.png" width="50%">

## Guardar el modelo ajustado

En este paso, guardamos el modelo ajustado tanto **localmente** como en el **Hub de Hugging Face** usando las credenciales de tu cuenta.


In [ ]:
trainer.save_model(output_dir)
trainer.push_to_hub(dataset_name=dataset_name)

## Cargar el modelo ajustado y ejecutar inferencia

Ahora, vamos a probar nuestro modelo ajustado cargando el **adaptador LoRA/QLoRA** y realizando **inferencia**. Empezaremos cargando el **modelo base** y luego le adjuntaremos el adaptador, creando el modelo ajustado final listo para evaluación.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

adapter_model = f"sergiopaniego/{output_dir}" # Replace with your HF username or organization

base_model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")

tokenizer = AutoTokenizer.from_pretrained(model_id)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Vamos a probar con un ejemplo del conjunto de prueba del dataset


In [ ]:
from datasets import load_dataset

dataset_name = 'AI-MO/NuminaMath-TIR'
test_dataset = load_dataset(dataset_name, split='test[:1%]')
test_dataset = test_dataset.map(make_conversation)
test_dataset = test_dataset.remove_columns(['messages', 'problem'])
test_dataset[0]['prompt']

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

[{'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant  first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process is enclosed strictly within <think> and </think> tags. After closing </think>, the assistant MUST provide the final answer in plain text.',
  'role': 'system'},
 {'content': "In 1988, a person's age was equal to the sum of the digits of their birth year. How old was this person?",
  'role': 'user'}]

Primero comprobemos cuál es la salida del modelo base, sin el adaptador.


In [ ]:
messages = test_dataset[0]['prompt']
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(
    **model_inputs,
    max_new_tokens=256
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

To solve this problem, let's denote the birth year of the person as \(Y\) (where \(Y\) is a four-digit number) and their age in 1988 as \(A\). According to the given condition, their age in 1988 is equal to the sum of the digits of their birth year. 

Since we're looking at the year 1988, the person would be \(1988 - Y\) years old in that year. Given the condition:

\[1988 - Y = \text{sum of the digits of } Y\]

Let's break down the possible range for \(Y\). Since the person's age must be less than or equal to 100 (as the sum of the digits of any four-digit number cannot exceed 36), \(Y\) must be between 1989 and 2088.

We can systematically check each year in this range to find when the condition holds true. However, considering the constraint on age, we can narrow our search significantly. For example, if \(Y\) were 1990, the sum of its digits would be 18, which is not a reasonable age. We need


El modelo base no generó trazas de razonamiento ni proporcionó una respuesta correcta. Carguemos ahora el modelo ajustado y comprobemos su rendimiento.


In [ ]:
fine_tuned_model = PeftModel.from_pretrained(base_model, adapter_model)

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/162M [00:00<?, ?B/s]

In [ ]:
text = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

generated_ids = fine_tuned_model.generate(
    **model_inputs,
    max_new_tokens=256
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

<think> I need to find a birth year where the sum of its digits equals the person's age in 1988 </think>

The person would have been born in 1979, since 1+9+7+9 = 26 and 26 is the age in 1988

answer: 26


¡La respuesta final es correcta!


## Inferencia y Servicio con vLLM

Puedes usar modelos Transformer con **vLLM** para servirlos en aplicaciones reales. Aprende más [aquí](https://blog.vllm.ai/2025/04/11/transformers-backend.html).


### Subir el Modelo Fusionado (para entrenamiento con LoRA o QLoRA)

Para servir el modelo mediante **vLLM**, el repositorio debe contener el modelo fusionado (modelo base + adaptador LoRA). Por tanto, es necesario subirlo primero.


In [ ]:
model_merged = fine_tuned_model.merge_and_unload()

save_dir = f"{output_dir}-merged"

model_merged.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

('Qwen2-7B-Instruct-GRPO-merged/tokenizer_config.json',
 'Qwen2-7B-Instruct-GRPO-merged/special_tokens_map.json',
 'Qwen2-7B-Instruct-GRPO-merged/chat_template.jinja',
 'Qwen2-7B-Instruct-GRPO-merged/vocab.json',
 'Qwen2-7B-Instruct-GRPO-merged/merges.txt',
 'Qwen2-7B-Instruct-GRPO-merged/added_tokens.json',
 'Qwen2-7B-Instruct-GRPO-merged/tokenizer.json')

In [ ]:
model_merged.push_to_hub(f"sergiopaniego/{output_dir}-merged") # Replace with your HF username or organization
tokenizer.push_to_hub(f"sergiopaniego/{output_dir}-merged") # Replace with your HF username or organization

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00004.safetensors:   0%|          |  612kB / 4.93GB            

  ...0003-of-00004.safetensors:   0%|          |  611kB / 4.33GB            

  ...0001-of-00004.safetensors:   1%|1         | 50.3MB / 4.88GB            

...

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...RPO-merged/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

CommitInfo(commit_url='https://huggingface.co/sergiopaniego/Qwen2-7B-Instruct-GRPO-merged/commit/b20988444532e79a6915f0b2b6002b5acc2b53e1', commit_message='Upload tokenizer', commit_description='', oid='b20988444532e79a6915f0b2b6002b5acc2b53e1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sergiopaniego/Qwen2-7B-Instruct-GRPO-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='sergiopaniego/Qwen2-7B-Instruct-GRPO-merged'), pr_revision=None, pr_num=None)

### Realizar Inferencia con vLLM

Usa **vLLM** para ejecutar tu modelo y generar texto de forma eficiente en tiempo real. Esto te permite probar y desplegar tus modelos ajustados con baja latencia y alto rendimiento.


In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import torch

llm = LLM(
    model=f"sergiopaniego/{output_dir}-merged", # Replace with your HF username or organization
    model_impl="transformers",                  # Select the transformers model implementation
    max_model_len=256,                         # Reduced for efficiency
    dtype=torch.float16
)
hf_tokenizer = AutoTokenizer.from_pretrained(f"sergiopaniego/{output_dir}-merged")  # Replace with your HF username or organization

INFO 12-11 15:56:09 [utils.py:253] non-default args: {'dtype': torch.float16, 'max_model_len': 256, 'disable_log_stats': True, 'model_impl': 'transformers', 'model': 'sergiopaniego/Qwen2-7B-Instruct-GRPO-merged'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


INFO 12-11 15:56:37 [model.py:631] Resolved architecture: TransformersForCausalLM
WARNING 12-11 15:56:37 [model.py:1971] Casting torch.bfloat16 to torch.float16.
INFO 12-11 15:56:37 [model.py:1745] Using max model len 256
INFO 12-11 15:56:40 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 12-11 15:56:43 [system_utils.py:103] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 12-11 15:57:36 [llm.py:352] Supported tasks: ['generate']


In [ ]:
messages = test_dataset[0]['prompt']
# Alternatively, use llm.chat()
prompt = hf_tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

outputs = llm.generate(
    {"prompt": prompt},
    sampling_params=SamplingParams(max_tokens=256),
)

for o in outputs:
    generated_text = o.outputs[0].text
    print(generated_text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

<think> 1988 birth year implies the person was born either in 1979, 1980, 1981, etc. Looking for the one where sum of digits equals age </think>

The birth year 1979 gives sum of digits 1+9+7+9 = 26

The person was 26 years old in 1988.

Answer: The person was 26 years old.
